In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# from handuflow import SystemConfigurator
# from handuflow import ValidationRunner
from handuflow import Orchestrator

builder = (
    SparkSession.builder.appName("HanduFLOW")
    .master("local[*]")
    .enableHiveSupport()
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 16:49:51 WARN Utils: Your hostname, handu-debian-portable, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface enp10s0)
26/09/20 16:49:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/handu/Documents/handuflow/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/handu/.ivy2.5.2/cache
The jars for the packages stored in: /home/handu/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8870d249-5bc5-4e68-9824-4e4658af5d92;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 79ms :: artifacts dl 3m

In [2]:
delta_table = DeltaTable.forName(
    spark,
    "spark_catalog.demo.employee",
)

latest_version = delta_table.history(1).select("version").first()

print(latest_version)

26/09/20 16:49:56 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/20 16:49:56 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore handu@127.0.1.1


Row(version=1)


In [3]:
spark.sql("show catalogs").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [4]:
spark.sql("show databases").show()

+-----------+
|  namespace|
+-----------+
|    default|
|       demo|
|    staging|
|target_test|
+-----------+



In [5]:
spark.sql(
    "select * from spark_catalog.staging.t_stg__targettest__parallel_customers__do_not_delete"
).show(truncate=False)

26/09/20 16:50:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-------+----+-------+-----------------------------+------------------------------------+----------------------------------------------------------------+-------------+--------------------------+
|id |name   |age |salary |event_ts                     |__x_run_id                          |__x_row_hash                                                    |__x_load_type|__x_ingestion_date        |
+---+-------+----+-------+-----------------------------+------------------------------------+----------------------------------------------------------------+-------------+--------------------------+
|3  |Charlie|28  |58000.0|2026-07-05 10:15:30.123+05:30|48d2cf1b-7fd0-4526-bd64-4223b187f932|c43b640e6ab164b019c8fd757beb913e0c023fc5cb58999c4fabe946ae6df8a3|FULL_LOAD    |2026-09-20 16:48:40.969804|
|6  |Frank  |40  |85000.0|2026-07-20 23:59:59.999+05:30|48d2cf1b-7fd0-4526-bd64-4223b187f932|d737d81c17451108cbd2be91dd2058995b12d9d47386bbfde06242b6fe7eb580|FULL_LOAD    |2026-09-20 16:48:40.969804|


In [6]:
spark.sql(
    """select * from spark_catalog.target_test.parallel_customers
    """
     
).show(truncate=False)

+---+-------+----+-------+-----------------------------+------------------------------------+----------------------------------------------------------------+-------------+--------------------------+
|id |name   |age |salary |event_ts                     |__x_run_id                          |__x_row_hash                                                    |__x_load_type|__x_ingestion_date        |
+---+-------+----+-------+-----------------------------+------------------------------------+----------------------------------------------------------------+-------------+--------------------------+
|3  |Charlie|28  |58000.0|2026-07-05 10:15:30.123+05:30|48d2cf1b-7fd0-4526-bd64-4223b187f932|c43b640e6ab164b019c8fd757beb913e0c023fc5cb58999c4fabe946ae6df8a3|FULL_LOAD    |2026-09-20 16:48:40.969804|
|6  |Frank  |40  |85000.0|2026-07-20 23:59:59.999+05:30|48d2cf1b-7fd0-4526-bd64-4223b187f932|d737d81c17451108cbd2be91dd2058995b12d9d47386bbfde06242b6fe7eb580|FULL_LOAD    |2026-09-20 16:48:40.969804|


In [7]:
spark.sql(
    """select * from spark_catalog.target_test.parallel_customers
    except

    select * from spark_catalog.staging.t_stg__targettest__parallel_customers__do_not_delete
    """
     
).show(truncate=False)

+---+----+---+------+--------+----------+------------+-------------+------------------+
|id |name|age|salary|event_ts|__x_run_id|__x_row_hash|__x_load_type|__x_ingestion_date|
+---+----+---+------+--------+----------+------------+-------------+------------------+
+---+----+---+------+--------+----------+------------+-------------+------------------+



In [8]:
spark.sql(
    """
    show TBLPROPERTIES spark_catalog.target_test.parallel_customers

    """
).show(truncate=False)

+-----------------------+---------+
|key                    |value    |
+-----------------------+---------+
|delta.minReaderVersion |1        |
|delta.minWriterVersion |2        |
|handuflow.loadType     |FULL_LOAD|
|handuflow.sourceVersion|1        |
+-----------------------+---------+



In [9]:
spark.sql(
    """
    describe history  spark_catalog.demo.employee

    """
).show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                         |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------+----+--------+---------+